In [8]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import pipeline
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [9]:
df = pd.read_csv('/content/Reviews.csv')

df = df[['Text', 'Score']].dropna()

print(df.head())

                                                Text  Score
0  I have bought several of the Vitality canned d...      5
1  Product arrived labeled as Jumbo Salted Peanut...      1
2  This is a confection that has been around a fe...      4
3  If you are looking for the secret ingredient i...      2
4  Great taffy at a great price.  There was a wid...      5


In [10]:
def sentiment(score):
    if score <= 2:
        return 0
    elif score == 3:
        return 1
    else:
        return 2

df['Sentiment'] = df['Score'].apply(sentiment)

print(df.head())

                                                Text  Score  Sentiment
0  I have bought several of the Vitality canned d...      5          2
1  Product arrived labeled as Jumbo Salted Peanut...      1          0
2  This is a confection that has been around a fe...      4          2
3  If you are looking for the secret ingredient i...      2          0
4  Great taffy at a great price.  There was a wid...      5          2


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    df['Text'], df['Sentiment'],
    test_size=0.2, random_state=42
)

X_train = X_train[:1000]
X_test = X_test[:300]
y_train = y_train[:1000]
y_test = y_test[:300]

In [12]:
def clean(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    return text.lower()

X_train = X_train.apply(clean)
X_test = X_test.apply(clean)

In [13]:
!pip install transformers datasets -q

In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

train_data = Dataset.from_dict({
    "text": X_train.tolist(),
    "labels": y_train.tolist()
})

test_data = Dataset.from_dict({
    "text": X_test.tolist(),
    "labels": y_test.tolist()
})

def tokenize(data):
    return tokenizer(data["text"], truncation=True, padding="max_length", max_length=128)

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=3
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


In [ ]:
tokenizer_lstm = Tokenizer(num_words=5000)
tokenizer_lstm.fit_on_texts(X_train)

X1 = pad_sequences(tokenizer_lstm.texts_to_sequences(X_train), maxlen=100)
X2 = pad_sequences(tokenizer_lstm.texts_to_sequences(X_test), maxlen=100)

lstm = Sequential([
    Embedding(5000, 64),
    LSTM(64),
    Dense(3, activation='softmax')
])

lstm.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

lstm.fit(X1, y_train, epochs=2, batch_size=32)

In [ ]:
bert_pred = trainer.predict(test_data).predictions.argmax(axis=1)
lstm_pred = lstm.predict(X2).argmax(axis=1)

print("BERT Accuracy:", accuracy_score(y_test, bert_pred))
print("LSTM Accuracy:", accuracy_score(y_test, lstm_pred))